In [8]:
from pykrx import stock
from datetime import datetime, timedelta

# 오늘 날짜와 60일 전 날짜 계산
end_date = datetime.now().strftime("%Y%m%d")
start_date = (datetime.now() - timedelta(days=60)).strftime("%Y%m%d")

# SK하이닉스(000660) 종가 데이터만 가져오기
df = stock.get_market_ohlcv(start_date, end_date, "000660")["종가"]

print("SK하이닉스 60일치 종가 데이터:")
print(df)


SK하이닉스 60일치 종가 데이터:
날짜
2025-04-25    184400
2025-04-28    182000
2025-04-29    180800
2025-04-30    177500
2025-05-02    186000
2025-05-07    190800
2025-05-08    190300
2025-05-09    190100
2025-05-12    195000
2025-05-13    198500
2025-05-14    206000
2025-05-15    200500
2025-05-16    204500
2025-05-19    199400
2025-05-20    202000
2025-05-21    200500
2025-05-22    196900
2025-05-23    200000
2025-05-26    203000
2025-05-27    202500
2025-05-28    208000
2025-05-29    212000
2025-05-30    204500
2025-06-02    207500
2025-06-04    217500
2025-06-05    224500
2025-06-09    229000
2025-06-10    230500
2025-06-11    240000
2025-06-12    235500
2025-06-13    235500
2025-06-16    248000
2025-06-17    249000
2025-06-18    246500
2025-06-19    246000
2025-06-20    257000
2025-06-23    259500
2025-06-24    278500
Name: 종가, dtype: int64


In [1]:
from pykrx import stock
from datetime import datetime, timedelta
import json

end_date = datetime.now().strftime("%Y%m%d")
start_date = (datetime.now() - timedelta(days=60)).strftime("%Y%m%d")
df = stock.get_market_ohlcv(start_date, end_date, "000660")

df_reset = df.reset_index()
data = [
{
    "date": row["날짜"].strftime("%Y-%m-%d"),
    "close": int(row["종가"])
}
for _, row in df_reset.iterrows()
]

with open("public/stock_data.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

In [ ]:
import os
import glob
import pandas as pd
import psycopg2
from psycopg2 import sql
from dotenv import load_dotenv
from datetime import datetime

# 🔐 환경 변수 로드
load_dotenv('stock.env')

# 📦 DB 연결 함수
def get_db_connection():
    return psycopg2.connect(
        host=os.getenv("DB_HOST"),
        dbname=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        port=os.getenv("DB_PORT")
    )

# 🔧 Normalize DataFrame to standard schema
STANDARD_COLUMNS = [
    'ticker', 'price_date', 'publisher_name', 'title', 'summary',
    'content', 'url', 'sentiment', 'sentiment_score', 'published_at', 'keywords'
]

def normalize_df(df: pd.DataFrame, ticker: str = None) -> pd.DataFrame:
    df = df.copy()

    # Ensure ticker
    if ticker:
        df['ticker'] = ticker
    if 'ticker' not in df.columns:
        df['ticker'] = ticker or None

    # Parse dates
    if 'published_at' in df.columns:
        df['published_at'] = pd.to_datetime(df['published_at'], errors='coerce')
    else:
        df['published_at'] = pd.NaT
    df['price_date'] = df['published_at'].dt.date

    # Fill missing mandatory columns
    for col in STANDARD_COLUMNS:
        if col not in df.columns:
            df[col] = None

    # Reorder
    return df[STANDARD_COLUMNS]

# 🛠 Publisher and Keyword helpers

def get_or_create_publisher(cur, name: str) -> int:
    cur.execute(
        "WITH ins AS ("
        " INSERT INTO publisher (name) VALUES (%s) ON CONFLICT (name) DO NOTHING RETURNING publisher_id)"
        " SELECT publisher_id FROM ins UNION SELECT publisher_id FROM publisher WHERE name = %s;",
        (name, name)
    )
    return cur.fetchone()[0]


def get_or_create_keyword(cur, word: str) -> int:
    cur.execute(
        "WITH ins AS ("
        " INSERT INTO keyword (word) VALUES (%s) ON CONFLICT (word) DO NOTHING RETURNING keyword_id)"
        " SELECT keyword_id FROM ins UNION SELECT keyword_id FROM keyword WHERE word = %s;",
        (word, word)
    )
    return cur.fetchone()[0]

# 🚀 Main pipeline: read Excels, normalize, insert

def pipeline_load_folder(folder_path: str):
    conn = get_db_connection()
    cur = conn.cursor()
    total_inserted = 0

    # iterate Excel files
    files = glob.glob(os.path.join(folder_path, '*.xlsx'))
    for file in files:
        # derive ticker from filename: ex SK하이닉스_000660.xlsx
        fname = os.path.basename(file)
        parts = os.path.splitext(fname)[0].split('_')
        ticker = parts[-1] if parts[-1].isdigit() else None

        df_raw = pd.read_excel(file)
        df = normalize_df(df_raw, ticker=ticker)

        for _, row in df.iterrows():
            try:
                # ticker exist
                cur.execute("SELECT 1 FROM ticker WHERE ticker = %s", (row['ticker'],))
                if cur.fetchone() is None:
                    continue

                # stock_price exist
                cur.execute(
                    "SELECT 1 FROM stock_price WHERE ticker = %s AND price_date = %s",
                    (row['ticker'], row['price_date'])
                )
                if cur.fetchone() is None:
                    continue

                # publisher
                publisher_id = get_or_create_publisher(cur, row['publisher_name'] or 'Unknown')

                # news
                cur.execute(
                    "INSERT INTO news (ticker, price_date, publisher_id, title, summary, content, url, sentiment, sentiment_score, published_at)"
                    " VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)"
                    " ON CONFLICT (url) DO NOTHING RETURNING news_id;",
                    (
                        row['ticker'], row['price_date'], publisher_id,
                        row['title'], row['summary'], row['content'], row['url'],
                        row['sentiment'], row['sentiment_score'], row['published_at']
                    )
                )
                nid = cur.fetchone()
                if nid:
                    news_id = nid[0]
                    total_inserted += 1
                else:
                    cur.execute("SELECT news_id FROM news WHERE url = %s", (row['url'],))
                    news_id = cur.fetchone()[0]

                # keywords
                for kw in str(row['keywords']).split(','):
                    kw = kw.strip()
                    if not kw:
                        continue
                    kid = get_or_create_keyword(cur, kw)
                    cur.execute(
                        "INSERT INTO news_keyword (news_id, keyword_id) VALUES (%s, %s) ON CONFLICT DO NOTHING;",
                        (news_id, kid)
                    )

            except Exception as e:
                conn.rollback()
                continue

        # commit after each file
        conn.commit()

    cur.close()
    conn.close()
    print(f"✅ 총 {total_inserted}건 삽입 완료")